# Clean all datasets

This notebook cleans **all** CSV datasets in the project:

- **Removes missing values** (NaN, empty strings, "N/A", "null", etc.)
- **Drops columns not needed for training** (IDs, payment_mode, location, etc.)
- Keeps only rows where **text** (description/notes) and **category** (label) are present

**Output:** For each dataset we write `*_cleaned.csv` in the same folder.

## 0. Install dependencies (run once)

Run the cell below to install required packages. This notebook needs **pandas**; `re` and `pathlib` are built-in.

In [1]:
# --- This notebook only needs pandas ---
%pip install pandas

# --- For other project notebooks (e.g. random-forest, neural-network) install: ---
# %pip install pandas numpy matplotlib scikit-learn nltk ipykernel

Note: you may need to restart the kernel to use updated packages.


## 1. Setup and list all datasets

In [2]:
import pandas as pd
import re
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()

# Folders to look for CSV files (project root + kaggle_download)
SEARCH_DIRS = [PROJECT_ROOT, PROJECT_ROOT / "kaggle_download"]

def get_all_csv_files():
    """Find all CSV files to clean. Exclude .venv, _cleaned, and non-existent dirs."""
    seen = set()
    out = []
    for d in SEARCH_DIRS:
        if not d.exists():
            continue
        for f in d.glob("*.csv"):
            if ".venv" in str(f) or "_cleaned" in f.stem:
                continue
            key = (f.parent.name, f.name)
            if key in seen:
                continue
            seen.add(key)
            out.append(f)
    return sorted(out, key=lambda p: (str(p.parent), p.name))

ALL_CSV_FILES = get_all_csv_files()
print("Datasets to clean:", len(ALL_CSV_FILES))
for f in ALL_CSV_FILES:
    print(" -", f)

Datasets to clean: 8
 - /home/mukama/Documents/transaction-categorization-main/Personal_Finance_Dataset.csv
 - /home/mukama/Documents/transaction-categorization-main/budget_data.csv
 - /home/mukama/Documents/transaction-categorization-main/budgetwise_finance_dataset.csv
 - /home/mukama/Documents/transaction-categorization-main/budgetwise_synthetic_dirty.csv
 - /home/mukama/Documents/transaction-categorization-main/transactions_dataset.csv
 - /home/mukama/Documents/transaction-categorization-main/transactions_kaggle.csv
 - /home/mukama/Documents/transaction-categorization-main/kaggle_download/11 march 2025.csv
 - /home/mukama/Documents/transaction-categorization-main/kaggle_download/budget_data.csv


## 2. Cleaning configuration

In [3]:
# Columns to DROP (not needed for training)
COLUMNS_TO_DROP = [
    "transaction_id", "user_id", "payment_mode", "location",
    "id", "Id", "index", "Unnamed: 0",
    "Type", "transaction_type",  # optional: keep if you need Income/Expense
]

# Placeholder strings to treat as missing (will become NaN and then drop rows in required cols)
MISSING_VALUES = ["", "N/A", "n/a", "na", "null", "None", "...", "nan", "-"]

# Names that identify the "text" column (description for categorization)
TEXT_COLUMN_NAMES = ["notes", "Transaction Description", "Text", "description", "text", "details", "narration"]

# Names that identify the "category" column (label for training)
CATEGORY_COLUMN_NAMES = ["category", "Category", "Category Id", "categories", "label", "expense type"]

## 3. Cleaning functions

In [5]:
def is_missing(val):
    if pd.isna(val):
        return True
    s = str(val).strip()
    if s in MISSING_VALUES or s.lower() in [m.lower() for m in MISSING_VALUES]:
        return True
    return False


def clean_amount(value):
    """Convert amount to numeric: remove Rs., ₹, $, commas."""
    if pd.isna(value):
        return None
    s = str(value).strip().replace(",", "")
    s = re.sub(r"[Rs\.\s₹$]", "", s, flags=re.IGNORECASE)
    if not s:
        return None
    try:
        return float(s)
    except ValueError:
        return None


def find_text_and_category_columns(df):
    """Return (text_col, category_col)."""
    cols_lower = {c.lower(): c for c in df.columns}
    text_col = None
    for name in TEXT_COLUMN_NAMES:
        if name.lower() in cols_lower:
            text_col = cols_lower[name.lower()]
            break
    if text_col is None and len(df.columns) >= 1:
        text_col = df.columns[0]
    category_col = None
    for name in CATEGORY_COLUMN_NAMES:
        if name.lower() in cols_lower:
            category_col = cols_lower[name.lower()]
            break
    if category_col is None and len(df.columns) >= 2:
        category_col = df.columns[1]
    return text_col, category_col


def clean_one_dataset(df, path, columns_to_drop, missing_values):
    """
    Drop unneeded columns, replace placeholders with NaN, drop rows with NaN in required columns (text + category),
    clean amount if present.
    """
    df = df.copy()
    # Drop columns that exist
    to_drop = [c for c in columns_to_drop if c in df.columns]
    df = df.drop(columns=to_drop, errors="ignore")

    # Replace placeholder missing with NaN
    for col in df.columns:
        df[col] = df[col].apply(lambda x: pd.NA if is_missing(x) else x)

    text_col, category_col = find_text_and_category_columns(df)
    required = []
    if text_col and text_col in df.columns:
        required.append(text_col)
    if category_col and category_col in df.columns:
        required.append(category_col)
    if not required:
        required = list(df.columns[:2]) if len(df.columns) >= 2 else list(df.columns)

    df = df.dropna(subset=required)

    if "amount" in df.columns:
        df["amount"] = df["amount"].apply(clean_amount)
    if "Amount" in df.columns:
        df["Amount"] = df["Amount"].apply(clean_amount)

    return df.reset_index(drop=True)

IndentationError: expected an indented block after function definition on line 24 (3095963154.py, line 25)

## 4. Run cleaning on all datasets

In [ ]:
results = []
for path in ALL_CSV_FILES:
    try:
        raw = pd.read_csv(path)
        n_before = len(raw)
        cleaned = clean_one_dataset(raw, path, COLUMNS_TO_DROP, MISSING_VALUES)
        n_after = len(cleaned)
        out_name = path.stem + "_cleaned" + path.suffix
        out_path = path.parent / out_name
        cleaned.to_csv(out_path, index=False)
        results.append({"file": path.name, "before": n_before, "after": n_after, "output": out_path.name})
        print(f"{path.name}: {n_before} -> {n_after} rows, saved to {out_path}")
    except Exception as e:
        print(f"{path.name}: Error - {e}")
        results.append({"file": path.name, "error": str(e)})
print("Done.")

## 5. Summary and preview

In [ ]:
ok = [r for r in results if "error" not in r]
if ok:
    summary = pd.DataFrame(ok)
    display(summary)
    first = ok[0]
    out_path = next((p for p in ALL_CSV_FILES if p.name == first["file"]), None)
    if out_path is not None:
        out_file = out_path.parent / (out_path.stem + "_cleaned" + out_path.suffix)
        if out_file.exists():
            sample = pd.read_csv(out_file, nrows=5)
            print("Columns in cleaned data:", list(sample.columns))
            display(sample)
else:
    print("No datasets were cleaned successfully.")